# Swerve one-year bank: evidence audit

Target: 1,825 unique approved questions. Current coverage is not the target. This notebook verifies actual item counts, source bases and one-time reservations, using only Python standard-library modules.


In [1]:
from pathlib import Path
import importlib.util, json
root = Path.cwd()
while not (root / 'scripts/import-surveys.py').exists():
    if root.parent == root: raise RuntimeError('Open inside the Swerve checkout')
    root = root.parent
spec = importlib.util.spec_from_file_location('surveys', root / 'scripts/import-surveys.py')
surveys = importlib.util.module_from_spec(spec)
spec.loader.exec_module(surveys)
outputs, audit = surveys.build()
print(json.dumps(audit, indent=2))


{
  "version": "surveys-v2-unique",
  "sourceCommit": "4c1ff5e3aef1816ae04af63218015066e186c147",
  "yearGoal": 1825,
  "verifiedQuestions": 310,
  "remainingToYearGoal": 1515,
  "scheduledDays": 62,
  "scheduledSlots": 310,
  "distinctDailySets": 62,
  "repeatedQuestionSlots": 0,
  "unscheduledReserve": 0,
  "firstScheduledDay": "2026-10-19",
  "lastScheduledDay": "2026-12-19",
  "categories": {
    "Food": 42,
    "Flying": 18,
    "Screen": 37,
    "Everyday": 10,
    "Traditions": 13,
    "Personality": 28,
    "Music": 18,
    "Interests": 30,
    "Habits": 34,
    "Travel": 61,
    "Community": 19
  },
  "minValidAnswers": 213,
  "sourceCount": 15,
  "currentOrRepresentativeClaim": false,
  "limits": [
    "Historical source reference groups, not current Swerve player votes.",
    "Count-based and published-weighted items are distinguished; weights are never used to invent respondent counts.",
    "One survey item/concept/original wording may be reserved once only. Rewordings or 

In [2]:
ledger = json.loads(outputs['docs/content/survey-ledger.json'])
for item in ledger:
    assert item['validAnswers'] >= 200
    if item['source']['basis'] == 'published-weighted':
        assert all(o['count'] is None for o in item['options'])
        assert abs(sum(o['pct'] for o in item['options']) - 100) < 1e-7
    else:
        assert sum(o['count'] for o in item['options']) == item['validAnswers']
        assert all(o['pct'] == 100 * o['count'] / item['validAnswers'] for o in item['options'])
    if item['respondentCount'] is not None:
        assert item['validAnswers'] + item['missingAnswers'] == item['respondentCount']
for path, generated in outputs.items():
    assert (root / path).read_text() == generated, path
registry = json.loads(outputs['content/publication-registry.json'])
items = [q for d in registry['days'] for q in d['items']]
for field in ['questionId','conceptKey','wordingFingerprint']:
    assert len({q[field] for q in items}) == len(items)
print(f"{len(ledger)} verified items; {len(items)} scheduled once over {len(registry['days'])} days.")
print(f"{audit['unscheduledReserve']} spare; {audit['remainingToYearGoal']} more needed for a year.")


310 verified items; 310 scheduled once over 62 days.
0 spare; 1515 more needed for a year.


In [3]:
example = next(q for q in ledger if q['questionId'] == 'ntas0601b-r17')
assert example['validAnswers'] == 2694
assert example['options'][0]['label'] == 'Several times a day'
print(example['originalWording'], '/', example['originalSubheading'])
print('Unweighted respondent base:', example['validAnswers'])
for option in example['options']:
    print(option['cell'], option['label'], f"{option['pct']:.6f}%", 'person count:', option['count'])


How often do you travel in the week by the following modes / By personally driving a car, van or lorry
Unweighted respondent base: 2694
NTAS0601b!R17C3 Several times a day 26.980330% person count: None
NTAS0601b!R17C4 Once a day 12.084831% person count: None
NTAS0601b!R17C5 Several times a week 28.250655% person count: None
NTAS0601b!R17C6 Once a week 5.864063% person count: None
NTAS0601b!R17C7 Less than once a week 4.032280% person count: None
NTAS0601b!R17C8 Never 22.787841% person count: None


## Limits

The year goal is unfinished. Historical reference populations are not current Swerve voters. The Slovak student/friend convenience sample is not representative of all young people. Weighted shares are not converted into invented person counts; missingness beyond published community bases remains unknown. A 200-response floor is an editorial policy, not proof of representativeness. Exact wording fingerprints and reviewed concept keys detect specified duplicate identities; semantic paraphrase review still requires editorial judgment. Candidate and blocked collections are not approved inventory.
